In [70]:
import pandas as pd
import numpy as np
import json


# ============================================================
# 1. LOAD / PREPARE DATA
# ============================================================

# If df is already loaded, this line is not necessary.
# Uncomment only if you need to load the cleaned dataset again.

import pandas as pd
import numpy as np

df = pd.read_csv(
    "/content/final_cleaned_data.csv",
    low_memory=False
)

# Convert date columns
df["date_added"] = pd.to_datetime(
    df["date_added"],
    errors="coerce"
)

df["date_updated"] = pd.to_datetime(
    df["date_updated"],
    errors="coerce"
)

df["read_at"] = pd.to_datetime(
    df["read_at"],
    errors="coerce"
)

df["started_at"] = pd.to_datetime(
    df["started_at"],
    errors="coerce"
)

print(df[[
    "date_added",
    "date_updated",
    "read_at",
    "started_at"
]].dtypes)


# ------------------------------------------------------------
# Numeric columns
# ------------------------------------------------------------

df["target"] = pd.to_numeric(
    df["target"],
    errors="coerce"
)

df["rating"] = pd.to_numeric(
    df["rating"],
    errors="coerce"
)

df["num_pages"] = pd.to_numeric(
    df["num_pages"],
    errors="coerce"
)

df["average_rating"] = pd.to_numeric(
    df["average_rating"],
    errors="coerce"
)

df["publication_year"] = pd.to_numeric(
    df["publication_year"],
    errors="coerce"
)

if "original_publication_year" in df.columns:
    df["original_publication_year"] = pd.to_numeric(
        df["original_publication_year"],
        errors="coerce"
    )


# ============================================================
# 2. NORMALIZE AUTHOR IDs
# ============================================================

def normalize_author_id(value):

    if pd.isna(value):
        return None

    # Numeric values such as 38531.0 -> "38531"
    try:
        numeric_value = float(value)

        if numeric_value.is_integer():
            return str(int(numeric_value))

    except (ValueError, TypeError):
        pass

    value = str(value).strip()

    # Remove accidental .0 from strings such as "38531.0"
    if value.endswith(".0"):
        try:
            return str(int(float(value)))
        except (ValueError, TypeError):
            pass

    return value


def parse_author_list(value):

    if pd.isna(value):
        return []

    # Already a Python list
    if isinstance(value, list):

        result = []

        for author in value:

            normalized = normalize_author_id(author)

            if normalized is not None:
                result.append(normalized)

        # Remove duplicates while preserving order
        return list(dict.fromkeys(result))

    value = str(value).strip()

    if not value:
        return []

    import re

    # Example:
    # "['38531', '267687']"
    ids = re.findall(
        r"'([^']*)'",
        value
    )

    if not ids:
        ids = re.findall(
            r'"([^"]*)"',
            value
        )

    if not ids:
        # Handle formats such as:
        # [38531, 267687]
        value = value.strip("[]")
        ids = [
            x.strip()
            for x in value.split(",")
            if x.strip()
        ]

    result = []

    for author in ids:

        normalized = normalize_author_id(author)

        if normalized is not None:
            result.append(normalized)

    return list(dict.fromkeys(result))


# ------------------------------------------------------------
# Create author_id_list if necessary
# ------------------------------------------------------------

if "author_id_list" in df.columns:

    df["author_id_list"] = df["author_id_list"].apply(
        parse_author_list
    )

elif "author_ids" in df.columns:

    df["author_id_list"] = df["author_ids"].apply(
        parse_author_list
    )

elif "primary_author_id" in df.columns:

    df["author_id_list"] = df["primary_author_id"].apply(
        lambda x: (
            [normalize_author_id(x)]
            if normalize_author_id(x) is not None
            else []
        )
    )

else:

    df["author_id_list"] = [[] for _ in range(len(df))]


# ------------------------------------------------------------
# Create primary_author_id from normalized list
# ------------------------------------------------------------

df["primary_author_id"] = df["author_id_list"].apply(
    lambda x: x[0] if len(x) > 0 else pd.NA
).astype("string")


# ============================================================
# 3. NORMALIZE GENRE LIST
# ============================================================

def parse_genre_list(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return [
            str(x).strip()
            for x in value
            if str(x).strip()
        ]

    value = str(value).strip()

    if not value:
        return []

    import ast

    # Try Python-list format
    try:

        parsed = ast.literal_eval(value)

        if isinstance(parsed, list):

            return [
                str(x).strip()
                for x in parsed
                if str(x).strip()
            ]

    except Exception:
        pass

    # Fallback
    return [
        x.strip()
        for x in value.strip("[]").split(",")
        if x.strip()
    ]


if "canonical_genres_list" in df.columns:

    df["canonical_genres_list"] = df[
        "canonical_genres_list"
    ].apply(parse_genre_list)

elif "cleaned_genres" in df.columns:

    df["canonical_genres_list"] = df[
        "cleaned_genres"
    ].apply(parse_genre_list)

elif "genres_list" in df.columns:

    df["canonical_genres_list"] = df[
        "genres_list"
    ].apply(parse_genre_list)

else:

    df["canonical_genres_list"] = [[] for _ in range(len(df))]


# ============================================================
# 4. PUBLICATION YEAR FALLBACK
# ============================================================

# If publication_year is missing, use original_publication_year.

if "original_publication_year" in df.columns:

    df["publication_year_final"] = df[
        "publication_year"
    ]

    missing_year = df["publication_year_final"].isna()

    df.loc[
        missing_year,
        "publication_year_final"
    ] = df.loc[
        missing_year,
        "original_publication_year"
    ]

else:

    df["publication_year_final"] = df[
        "publication_year"
    ]


# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def page_bucket(pages):

    if pd.isna(pages):
        return None

    pages = float(pages)

    if pages < 200:
        return "<200"

    elif pages < 400:
        return "200-399"

    elif pages < 600:
        return "400-599"

    else:
        return "600+"


def rating_bucket(rating):

    if pd.isna(rating):
        return None

    rating = float(rating)

    if rating < 3.5:
        return "<3.5"

    elif rating < 4.0:
        return "3.5-3.99"

    elif rating < 4.5:
        return "4.0-4.49"

    else:
        return "4.5-5.0"


def year_bucket(year):

    if pd.isna(year):
        return None

    year = float(year)

    if year < 1980:
        return "Before 1980"

    elif year < 1990:
        return "1980-1989"

    elif year < 2000:
        return "1990-1999"

    elif year < 2010:
        return "2000-2009"

    elif year < 2015:
        return "2010-2014"

    else:
        return "2015+"


def make_rate_map(counter):

    result = []

    for key, values in counter.items():

        eligible = values["eligible"]
        converted = values["converted"]

        if eligible > 0:

            result.append({
                "key": key,
                "eligible": int(eligible),
                "converted": int(converted),
                "conversion_rate": converted / eligible
            })

    return result


# ============================================================
# 6. TEMPORAL FEATURE ENGINEERING
# ============================================================

def build_temporal_features(user_df):

    # --------------------------------------------------------
    # Sort by timestamp and book_id
    # --------------------------------------------------------

    user_df = user_df.sort_values(
        ["date_added", "book_id"],
        kind="mergesort"
    ).copy()

    results = []

    # --------------------------------------------------------
    # Historical state
    #
    # IMPORTANT:
    # We DO NOT immediately add a row after processing it.
    #
    # All rows having the same date_added are processed first.
    # Only after the complete timestamp group is processed do
    # we add those rows to history.
    # --------------------------------------------------------

    history = []

    # ========================================================
    # PROCESS ONE EXACT TIMESTAMP AT A TIME
    # ========================================================

    for current_date, timestamp_rows in user_df.groupby(
        "date_added",
        sort=True
    ):

        timestamp_rows = timestamp_rows.copy()

        # ----------------------------------------------------
        # Build mature history BEFORE this timestamp
        # ----------------------------------------------------

        mature_history = []

        for previous in history:

            previous_date = previous["date_added"]
            previous_target = previous["target"]

            if pd.isna(previous_date):
                continue

            if pd.isna(previous_target):
                continue

            if previous_target not in [0, 1]:
                continue

            # 90-day observation window must be complete
            if (
                previous_date + pd.Timedelta(days=90)
                <= current_date
            ):

                mature_history.append(previous)


        # ====================================================
        # CALCULATE FEATURES FOR EVERY ROW AT THIS TIMESTAMP
        # ====================================================

        timestamp_results = []

        for _, row in timestamp_rows.iterrows():

            # ------------------------------------------------
            # BASIC USER HISTORY
            # ------------------------------------------------

            previous_interactions = len(history)

            previous_eligible = len(mature_history)

            previous_conversions_90d = sum(
                int(x["target"])
                for x in mature_history
            )

            if previous_eligible > 0:

                user_conversion_rate_90d = (
                    previous_conversions_90d
                    / previous_eligible
                )

            else:

                user_conversion_rate_90d = np.nan


            # ------------------------------------------------
            # HISTORICAL USER RATING
            #
            # IMPORTANT:
            # rating = 0 means no actual user rating.
            # Therefore ignore rating <= 0.
            # ------------------------------------------------

            valid_ratings = []

            for previous in history:

                previous_rating = previous["rating"]

                if pd.isna(previous_rating):
                    continue

                if previous_rating <= 0:
                    continue

                # Rating must have been available at the
                # current timestamp.
                #
                # For an interaction rating, we only use it
                # if its date_updated is available and not
                # after the current date.
                rating_date = previous.get(
                    "date_updated",
                    pd.NaT
                )

                if pd.notna(rating_date):

                    if rating_date > current_date:
                        continue

                valid_ratings.append(
                    float(previous_rating)
                )

            if len(valid_ratings) > 0:

                user_avg_rating = (
                    sum(valid_ratings)
                    / len(valid_ratings)
                )

            else:

                user_avg_rating = np.nan


            # ------------------------------------------------
            # HISTORICAL READ PAGES
            #
            # A book contributes only if:
            #
            # read_at <= current_date
            #
            # This prevents future reading information leakage.
            # ------------------------------------------------

            read_pages = []

            for previous in history:

                previous_pages = previous["num_pages"]
                previous_read_at = previous["read_at"]

                if pd.isna(previous_pages):
                    continue

                if pd.isna(previous_read_at):
                    continue

                if previous_read_at > current_date:
                    continue

                # is_read, when available, should indicate
                # that the book was actually read.
                previous_is_read = previous.get(
                    "is_read",
                    np.nan
                )

                if pd.notna(previous_is_read):

                    if int(previous_is_read) != 1:
                        continue

                read_pages.append(
                    float(previous_pages)
                )

            if len(read_pages) > 0:

                user_avg_pages_read = (
                    sum(read_pages)
                    / len(read_pages)
                )

            else:

                user_avg_pages_read = np.nan


            # =================================================
            # HISTORICAL GENRE CONVERSION MAP
            # =================================================

            genre_counter = {}

            for previous in mature_history:

                previous_genres = previous[
                    "canonical_genres_list"
                ]

                for genre in previous_genres:

                    if genre not in genre_counter:

                        genre_counter[genre] = {
                            "eligible": 0,
                            "converted": 0
                        }

                    genre_counter[genre]["eligible"] += 1

                    genre_counter[genre]["converted"] += int(
                        previous["target"]
                    )

            genre_conversion_rates = make_rate_map(
                genre_counter
            )


            # =================================================
            # HISTORICAL AUTHOR CONVERSION MAP
            #
            # USE ALL AUTHORS, NOT ONLY PRIMARY AUTHOR
            # =================================================

            author_counter = {}

            for previous in mature_history:

                previous_authors = previous[
                    "author_id_list"
                ]

                for author in previous_authors:

                    author = normalize_author_id(author)

                    if author is None:
                        continue

                    if author not in author_counter:

                        author_counter[author] = {
                            "eligible": 0,
                            "converted": 0
                        }

                    author_counter[author]["eligible"] += 1

                    author_counter[author]["converted"] += int(
                        previous["target"]
                    )

            author_conversion_rates = make_rate_map(
                author_counter
            )


            # =================================================
            # PAGE GROUP CONVERSION MAP
            # =================================================

            page_counter = {}

            for previous in mature_history:

                bucket = page_bucket(
                    previous["num_pages"]
                )

                if bucket is None:
                    continue

                if bucket not in page_counter:

                    page_counter[bucket] = {
                        "eligible": 0,
                        "converted": 0
                    }

                page_counter[bucket]["eligible"] += 1

                page_counter[bucket]["converted"] += int(
                    previous["target"]
                )

            page_group_conversion_rates = make_rate_map(
                page_counter
            )


            # =================================================
            # BOOK AVERAGE-RATING GROUP CONVERSION MAP
            # =================================================

            rating_counter = {}

            for previous in mature_history:

                bucket = rating_bucket(
                    previous["average_rating"]
                )

                if bucket is None:
                    continue

                if bucket not in rating_counter:

                    rating_counter[bucket] = {
                        "eligible": 0,
                        "converted": 0
                    }

                rating_counter[bucket]["eligible"] += 1

                rating_counter[bucket]["converted"] += int(
                    previous["target"]
                )

            book_rating_group_conversion_rates = make_rate_map(
                rating_counter
            )


            # =================================================
            # PUBLICATION YEAR GROUP CONVERSION MAP
            # =================================================

            year_counter = {}

            for previous in mature_history:

                previous_year = previous[
                    "publication_year_final"
                ]

                bucket = year_bucket(
                    previous_year
                )

                if bucket is None:
                    continue

                if bucket not in year_counter:

                    year_counter[bucket] = {
                        "eligible": 0,
                        "converted": 0
                    }

                year_counter[bucket]["eligible"] += 1

                year_counter[bucket]["converted"] += int(
                    previous["target"]
                )

            publication_year_group_conversion_rates = (
                make_rate_map(year_counter)
            )


            # =================================================
            # CURRENT ROW
            # =================================================

            result_row = row.copy()

            result_row[
                "previous_interactions"
            ] = previous_interactions

            result_row[
                "previous_eligible"
            ] = previous_eligible

            result_row[
                "previous_conversions_90d"
            ] = previous_conversions_90d

            result_row[
                "user_conversion_rate_90d"
            ] = user_conversion_rate_90d

            result_row[
                "user_avg_rating"
            ] = user_avg_rating

            result_row[
                "user_avg_pages_read"
            ] = user_avg_pages_read

            result_row[
                "genre_conversion_rates"
            ] = json.dumps(
                genre_conversion_rates
            )

            result_row[
                "author_conversion_rates"
            ] = json.dumps(
                author_conversion_rates
            )

            result_row[
                "page_group_conversion_rates"
            ] = json.dumps(
                page_group_conversion_rates
            )

            result_row[
                "book_rating_group_conversion_rates"
            ] = json.dumps(
                book_rating_group_conversion_rates
            )

            result_row[
                "publication_year_group_conversion_rates"
            ] = json.dumps(
                publication_year_group_conversion_rates
            )

            timestamp_results.append(
                result_row
            )


        # ====================================================
        # ADD ALL ROWS FROM THIS TIMESTAMP TO HISTORY
        #
        # IMPORTANT:
        # This happens AFTER all rows at this timestamp have
        # received their features.
        # ====================================================

        results.extend(timestamp_results)

        for _, row in timestamp_rows.iterrows():

            history.append(
                row.to_dict()
            )


    return pd.DataFrame(results)


# ============================================================
# 7. SORT COMPLETE DATASET
# ============================================================

df = df.sort_values(
    ["user_id", "date_added", "book_id"],
    kind="mergesort"
).reset_index(drop=True)


# ============================================================
# 8. BUILD TEMPORAL DATASET USER BY USER
# ============================================================

print("Building temporal features...")

user_results = []

for user_id, user_df in df.groupby(
    "user_id",
    sort=False
):

    result = build_temporal_features(
        user_df
    )

    user_results.append(result)


training_data = pd.concat(
    user_results,
    ignore_index=True
)


# ============================================================
# 9. KEEP ONLY VALID SUPERVISED TARGETS
# ============================================================

training_data["target"] = pd.to_numeric(
    training_data["target"],
    errors="coerce"
)

training_data = training_data[
    training_data["target"].isin([0, 1])
].copy()


training_data["target"] = (
    training_data["target"]
    .astype(int)
)


# ============================================================
# 10. FINAL REQUIRED COLUMNS
# ============================================================

required_columns = [

    "user_id",
    "book_id",
    "date_added",

    "previous_interactions",
    "previous_eligible",
    "previous_conversions_90d",
    "user_conversion_rate_90d",

    "user_avg_rating",
    "user_avg_pages_read",

    "canonical_genres_list",
    "author_id_list",
    "primary_author_id",

    "genre_conversion_rates",
    "author_conversion_rates",
    "page_group_conversion_rates",
    "book_rating_group_conversion_rates",
    "publication_year_group_conversion_rates",

    "num_pages",
    "average_rating",
    "publication_year",

    "target"
]


missing = [
    column
    for column in required_columns
    if column not in training_data.columns
]


if missing:

    raise ValueError(
        f"Missing required columns: {missing}"
    )


# ============================================================
# 11. FINAL VALIDATION
# ============================================================

print("\n==========================================")
print("TEMPORAL DATASET CREATED SUCCESSFULLY")
print("==========================================")

print(
    "Training data shape:",
    training_data.shape
)

print(
    "Missing required columns:",
    missing
)

print(
    "Target distribution:"
)

print(
    training_data["target"].value_counts(
        dropna=False
    )
)

print(
    "\nNon-string primary author IDs:"
)

print(
    training_data[
        "primary_author_id"
    ]
    .dropna()
    .map(
        lambda x: not isinstance(x, str)
    )
    .sum()
)


print(
    "\nRows with author conversion history:"
)

print(
    (
        training_data[
            "author_conversion_rates"
        ]
        .fillna("")
        .astype(str)
        .str.len()
        > 2
    ).sum()
)


# ============================================================
# 12. CHECK SAME-TIMESTAMP BEHAVIOR
# ============================================================

duplicate_timestamps = (
    training_data
    .groupby(
        ["user_id", "date_added"]
    )
    .size()
)

duplicate_timestamps = duplicate_timestamps[
    duplicate_timestamps > 1
]

print(
    "\nUser/timestamp groups containing multiple books:",
    len(duplicate_timestamps)
)


# ============================================================
# 13. DISPLAY IMPORTANT COLUMNS
# ============================================================

display(
    training_data[
        [
            "user_id",
            "book_id",
            "date_added",
            "previous_interactions",
            "previous_eligible",
            "previous_conversions_90d",
            "user_conversion_rate_90d",
            "user_avg_rating",
            "user_avg_pages_read",
            "primary_author_id",
            "author_conversion_rates",
            "genre_conversion_rates",
            "page_group_conversion_rates",
            "book_rating_group_conversion_rates",
            "publication_year_group_conversion_rates",
            "target"
        ]
    ].head(10)
)


# ============================================================
# 14. SAVE FINAL DATASET
# ============================================================

output_path = (
    "/content/training_data_temporal_corrected.csv"
)

training_data.to_csv(
    output_path,
    index=False
)

print(
    "\nSaved final temporal training dataset to:"
)

print(output_path)

date_added      datetime64[ns]
date_updated    datetime64[ns]
read_at         datetime64[ns]
started_at      datetime64[ns]
dtype: object
Building temporal features...

TEMPORAL DATASET CREATED SUCCESSFULLY
Training data shape: (71387, 40)
Missing required columns: []
Target distribution:
target
0    47726
1    23661
Name: count, dtype: int64

Non-string primary author IDs:
0

Rows with author conversion history:
57862

User/timestamp groups containing multiple books: 24


,user_id,book_id,date_added,previous_interactions,previous_eligible,previous_conversions_90d,user_conversion_rate_90d,user_avg_rating,user_avg_pages_read,primary_author_id,author_conversion_rates,genre_conversion_rates,page_group_conversion_rates,book_rating_group_conversion_rates,publication_year_group_conversion_rates,target
0,00467c6d04c1d124b9d07bec3a74cf61,3225007,2014-05-19 21:04:49,0,0,0,NaN,NaN,NaN,38531,[],[],[],[],[],0
1,00467c6d04c1d124b9d07bec3a74cf61,6257480,2014-05-19 22:28:25,1,0,0,NaN,NaN,NaN,267687,[],[],[],[],[],0
2,00467c6d04c1d124b9d07bec3a74cf61,76684,2014-05-19 22:30:36,2,0,0,NaN,NaN,NaN,16667,[],[],[],[],[],0
3,00467c6d04c1d124b9d07bec3a74cf61,15798314,2014-08-12 19:13:33,3,0,0,NaN,NaN,NaN,32638,[],[],[],[],[],0
4,00467c6d04c1d124b9d07bec3a74cf61,25968028,2015-08-06 08:04:55,4,4,0,0.000,NaN,NaN,6581316,"[{""key"": ""38531"", ""eligible"": 1, ""converted"": ...","[{""key"": ""vampires"", ""eligible"": 1, ""converted...","[{""key"": ""200-399"", ""eligible"": 3, ""converted""...","[{""key"": ""3.5-3.99"", ""eligible"": 2, ""converted...","[{""key"": ""2000-2009"", ""eligible"": 3, ""converte...",0
5,00467c6d04c1d124b9d07bec3a74cf61,25667060,2016-06-01 14:42:26,5,5,0,0.000,NaN,NaN,8835824,"[{""key"": ""38531"", ""eligible"": 1, ""converted"": ...","[{""key"": ""vampires"", ""eligible"": 2, ""converted...","[{""key"": ""200-399"", ""eligible"": 4, ""converted""...","[{""key"": ""3.5-3.99"", ""eligible"": 2, ""converted...","[{""key"": ""2000-2009"", ""eligible"": 3, ""converte...",1
6,00467c6d04c1d124b9d07bec3a74cf61,25814053,2016-06-12 05:24:40,6,5,0,0.000,4.000000,184.000000,4715088,"[{""key"": ""38531"", ""eligible"": 1, ""converted"": ...","[{""key"": ""vampires"", ""eligible"": 2, ""converted...","[{""key"": ""200-399"", ""eligible"": 4, ""converted""...","[{""key"": ""3.5-3.99"", ""eligible"": 2, ""converted...","[{""key"": ""2000-2009"", ""eligible"": 3, ""converte...",1
7,00467c6d04c1d124b9d07bec3a74cf61,18775125,2016-07-01 11:23:33,7,5,0,0.000,3.500000,186.000000,2101598,"[{""key"": ""38531"", ""eligible"": 1, ""converted"": ...","[{""key"": ""vampires"", ""eligible"": 2, ""converted...","[{""key"": ""200-399"", ""eligible"": 4, ""converted""...","[{""key"": ""3.5-3.99"", ""eligible"": 2, ""converted...","[{""key"": ""2000-2009"", ""eligible"": 3, ""converte...",1
8,00467c6d04c1d124b9d07bec3a74cf61,23395776,2016-10-10 23:12:55,8,8,3,0.375,3.666667,185.333333,249141,"[{""key"": ""38531"", ""eligible"": 1, ""converted"": ...","[{""key"": ""vampires"", ""eligible"": 2, ""converted...","[{""key"": ""200-399"", ""eligible"": 4, ""converted""...","[{""key"": ""3.5-3.99"", ""eligible"": 2, ""converted...","[{""key"": ""2000-2009"", ""eligible"": 3, ""converte...",1
10,00467c6d04c1d124b9d07bec3a74cf61,17794090,2017-01-05 23:26:06,10,8,3,0.375,3.200000,204.800000,4860023,"[{""key"": ""38531"", ""eligible"": 1, ""converted"": ...","[{""key"": ""vampires"", ""eligible"": 2, ""converted...","[{""key"": ""200-399"", ""eligible"": 4, ""converted""...","[{""key"": ""3.5-3.99"", ""eligible"": 2, ""converted...","[{""key"": ""2000-2009"", ""eligible"": 3, ""converte...",1



Saved final temporal training dataset to:
/content/training_data_temporal_corrected.csv


In [73]:
training_data.shape

(71387, 40)

In [74]:
training_data.head()

,user_id,book_id,is_read,rating,date_added,date_updated,read_at,started_at,reading_duration_days,title,...,previous_eligible,previous_conversions_90d,user_conversion_rate_90d,user_avg_rating,user_avg_pages_read,genre_conversion_rates,author_conversion_rates,page_group_conversion_rates,book_rating_group_conversion_rates,publication_year_group_conversion_rates
0,00467c6d04c1d124b9d07bec3a74cf61,3225007,0,0,2014-05-19 21:04:49,2014-05-30 15:29:02,NaT,NaT,NaN,"Stakes & Stilettos (Immortality Bites, #4)",...,0,0,NaN,NaN,NaN,[],[],[],[],[]
1,00467c6d04c1d124b9d07bec3a74cf61,6257480,0,0,2014-05-19 22:28:25,2017-07-28 21:58:18,NaT,NaT,NaN,"Obsession Untamed (Feral Warriors, #2)",...,0,0,NaN,NaN,NaN,[],[],[],[],[]
2,00467c6d04c1d124b9d07bec3a74cf61,76684,0,0,2014-05-19 22:30:36,2014-08-18 14:33:36,NaT,NaT,NaN,"I, Robot (Robot, #0.1)",...,0,0,NaN,NaN,NaN,[],[],[],[],[]
3,00467c6d04c1d124b9d07bec3a74cf61,15798314,0,0,2014-08-12 19:13:33,2015-02-24 17:05:39,NaT,NaT,NaN,"Born to Darkness (Fighting Destiny, #1)",...,0,0,NaN,NaN,NaN,[],[],[],[],[]
4,00467c6d04c1d124b9d07bec3a74cf61,25968028,0,0,2015-08-06 08:04:55,2016-03-26 01:44:21,NaT,NaT,NaN,The Eyes of The Sun (The Eyes of The Sun #1),...,4,0,0.0,NaN,NaN,"[{""key"": ""vampires"", ""eligible"": 1, ""converted...","[{""key"": ""38531"", ""eligible"": 1, ""converted"": ...","[{""key"": ""200-399"", ""eligible"": 3, ""converted""...","[{""key"": ""3.5-3.99"", ""eligible"": 2, ""converted...","[{""key"": ""2000-2009"", ""eligible"": 3, ""converte..."


In [75]:
from google.colab import files
files.download('/content/training_data_temporal_corrected.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>